In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Tickers in the same order you listed
tickers = [
    "MSFT","AMZN","GOOG","SNOW","ANET","ASML","LRCX","AVGO","SMH",
    "NEE","CEG","AES","EXC","AEP","BWXT","LEU","XYL","AWK","CWCO","WM",
    "HASI","ETN","PWR","HUBB","VRT","TT","WMT","CTAS","LEN"
]

# 2. Download 10-year daily history (auto-adjusted closes)
print("Downloading data …")
prices = yf.download(tickers, period="10y", interval="1d")["Close"].dropna()
print(f"History shape: {prices.shape}")

# 3. Compute daily log-returns
log_ret = np.log(prices / prices.shift(1)).dropna()

# 4. Equal-weight vector
w = np.ones(len(tickers)) / len(tickers)

# 5. Portfolio mean log-return and covariance
mu = log_ret.mean().values                 # (n, )
Sigma = log_ret.cov().values               # (n,n)

# 6. Monte-Carlo parameters
days_per_year = 252
years = 10
total_days = years * days_per_year
n_paths = 10_000
dt = 1 / days_per_year

# 7. Cholesky decomposition for correlated shocks
L = np.linalg.cholesky(Sigma)

# 8. Simulate 10 000 paths
print("Simulating 10 000 paths …")
Z = np.random.normal(size=(total_days, n_paths, len(tickers)))
shocks = Z @ L.T                             # correlate the shocks
# cumulative log-return for each asset
log_paths = (mu - 0.5 * np.diag(Sigma)) * dt + shocks * np.sqrt(dt)
log_paths = np.cumsum(log_paths, axis=0)

# 9. Portfolio log-return (equal-weight)
port_log_ret = (log_paths @ w)               # shape: (total_days, n_paths)

# 10. Convert to dollar levels
initial = 500_000
port_value = initial * np.exp(port_log_ret)  # shape: (total_days, n_paths)

# 11. Plot every path
t_vec = np.linspace(0, years, total_days)
plt.figure(figsize=(10, 6))
plt.plot(t_vec, port_value, lw=0.3, alpha=0.4, color="steelblue")
plt.title("10 000 Monte-Carlo paths\n30-asset equal-weight portfolio, 10-year hold, $500 k start")
plt.xlabel("Years")
plt.ylabel("Portfolio value ($)")
plt.yscale("log")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# 12. Summary statistics at the 10-year horizon
terminal = port_value[-1, :]
print("\n10-year horizon statistics (equal-weight):")
print(f"Mean terminal value: ${terminal.mean():,.0f}")
print(f"Median terminal value: ${np.median(terminal):,.0f}")
print(f"Std-dev terminal value: ${terminal.std():,.0f}")
print(f"5-th %-ile: ${np.percentile(terminal, 5):,.0f}")
print(f"95-th %-ile: ${np.percentile(terminal, 95):,.0f}")
print(f"Probability final value < $500 k: {(terminal < 500000).mean():.1%}")

/tmp/ipython-input-1589045133.py:19: FutureWarning: YF.download() has changed argument auto_adjust default to True
  prices = yf.download(tickers, period="10y", interval="1d")["Close"].dropna()
[*********************100%***********************]  29 of 29 completed


History shape: (970, 29)
Simulating 10 000 paths …
